In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/depi_scraping"

Mounted at /content/drive


In [ ]:
import json, re, time
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo
import requests

CONTACT = "your.ayaelgrwany18@gmail.com"
SITE = "xprs"
BASE_URL = "https://myxprs.com"

CATEGORIES = {
    "mobile-phones-egypt": "phone",
    "32-inch-tv": "tv", "43-inch-tv": "tv", "55-inch-tv": "tv", "65-inch-tv": "tv", "75-inch-tv": "tv",
    "85-inch-tv": "tv", "ultra-hd-4k-tvs": "tv", "ultra-hd-8k-tvs": "tv", "neo-qled-tvs": "tv",
}

session = requests.Session()
session.headers.update({"User-Agent": f"DEPI-Student-PriceBot/1.0 (educational project; contact: {CONTACT})"})

In [ ]:
def fetch_json(url):
    time.sleep(1)
    r = session.get(url, timeout=30)
    if r.status_code == 403:
        raise SystemExit("XPRS blocked us (403) - stop")
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()

In [ ]:
def to_price(value):
    if value is None:
        return None
    price = float(value)
    return price if price > 0 else None


def discount_pct(sale, original):
    if sale and original and original > sale:
        return round(100 * (1 - sale / original))
    return None

In [ ]:
def to_offers(product, category_site, snapshot_date, scraped_at):
    offers = []
    for v in product["variants"]:
        sku = (v.get("sku") or "").strip()
        is_ean = bool(re.fullmatch(r"\d{13}", sku))
        sale = to_price(v.get("price"))
        original = to_price(v.get("compare_at_price"))
        if not (sale and original and original > sale):
            original = None
        title = product["title"]
        if v["title"] != "Default Title":
            title += " - " + v["title"]
        offers.append({
            "site": SITE,
            "site_product_id": f'{product["handle"]}:{v["id"]}',
            "snapshot_date": snapshot_date,
            "scraped_at": scraped_at,
            "url": f'{BASE_URL}/products/{product["handle"]}?variant={v["id"]}',
            "title": title,
            "brand_raw": product.get("vendor"),
            "model_raw": sku if sku and not is_ean else None,
            "ean": sku if is_ean else None,
            "category_site": category_site,
            "category": CATEGORIES[category_site],
            "sale_price": sale,
            "original_price": original,
            "discount_pct": discount_pct(sale, original),
            "currency": "EGP",
            "stock_text": None,
            "in_stock": v.get("available"),
            "seller": None,
            "source_format": "json",
            "raw_file": None,
        })
    return offers

In [ ]:
now = datetime.now(timezone.utc)
snapshot_date = now.astimezone(ZoneInfo("Africa/Cairo")).strftime("%Y-%m-%d")
scraped_at = now.strftime("%Y-%m-%dT%H:%M:%SZ")

offers = []
for handle in CATEGORIES:
    for page in range(1, 41):
        url = f"{BASE_URL}/collections/{handle}/products.json?limit=250&page={page}"
        data = fetch_json(url)
        if data is None:
            print(handle, "-> not found, skipped")
            break
        products = data.get("products", [])
        if not products:
            break
        for p in products:
            offers += to_offers(p, handle, snapshot_date, scraped_at)
        print(handle, "page", page, "->", len(products), "products")

offers = list({o["site_product_id"]: o for o in offers}.values())
print("total offers:", len(offers))

mobile-phones-egypt page 1 -> 115 products
32-inch-tv page 1 -> 4 products
43-inch-tv page 1 -> 4 products
55-inch-tv page 1 -> 2 products
65-inch-tv page 1 -> 6 products
75-inch-tv page 1 -> 1 products
85-inch-tv page 1 -> 2 products
ultra-hd-4k-tvs page 1 -> 6 products
ultra-hd-8k-tvs page 1 -> 1 products
neo-qled-tvs page 1 -> 3 products
total offers: 357


In [ ]:
assert offers, "No offers - check the site"
path = Path(OUT_DIR) / f"xprs_offers_{snapshot_date}.json"
path.write_text(json.dumps(offers, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path)

from collections import Counter
print("per category:", dict(Counter(o["category"] for o in offers)))
print("no price:", sum(1 for o in offers if not o["sale_price"]))
print("in stock:", sum(1 for o in offers if o["in_stock"]))

saved: /content/drive/MyDrive/depi_scraping/xprs_offers_2026-10-29.json
per category: {'phone': 336, 'tv': 21}
no price: 0
in stock: 175
